In [ ]:
# 1. Environment Setup
!pip install transformers torch einops lancedb

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import numpy as np
import polars as pl
import lancedb
import pyarrow as pa
from typing import List

print("Environment Setup Complete")

In [ ]:
# 2. Model Loading & Patching
from transformers import AutoConfig, AutoModelForMaskedLM, AutoTokenizer

model_name = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

# CRITICAL: Monkey-patch intermediate_size to 4096 to fix SwiGLU activation mismatch
print(f"Loading model: {model_name} with patched configuration...")
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
config.intermediate_size = 4096 

model = AutoModelForMaskedLM.from_pretrained(model_name, config=config, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded successfully on {device}")

In [ ]:
# 3. Tokenization & Embedding Logic

def get_6mers(sequence: str):
    """
    Sliding window 6-mer tokenization.
    Example: 'ATGCGT...' -> 'ATGCGT TGCGTG GCGTGC ...'
    """
    k = 6
    return " ".join([sequence[i:i+k] for i in range(len(sequence) - k + 1)])

def get_embeddings(sequences: List[str], batch_size: int = 8):
    """
    Generates 512D embeddings and zero-pads them to 768D.
    """
    embeddings = []
    
    # Process in batches
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]
        
        # Tokenize (using 6-mer window)
        tokenized_batch = [get_6mers(seq) for seq in batch]
        
        # Tokenize for model input
        inputs = tokenizer(tokenized_batch, return_tensors="pt", padding=True, truncation=True, max_length=1000)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            
        # Extract last hidden state (mean pooling over sequence length, excluding special tokens if desired)
        # Using mean of all tokens for simplicity as a sequence embedding
        hidden_states = outputs.hidden_states[-1] # Shape: (batch, seq_len, 512)
        
        # Mean pooling
        # Mask out padding tokens
        attention_mask = inputs['attention_mask'].unsqueeze(-1)
        mean_embeddings = torch.sum(hidden_states * attention_mask, dim=1) / torch.sum(attention_mask, dim=1)
        
        mean_embeddings = mean_embeddings.cpu().numpy() # Shape: (batch, 512)
        
        # Zero-pad to 768D
        padded_embeddings = np.zeros((len(batch), 768), dtype=np.float32)
        padded_embeddings[:, :512] = mean_embeddings
        
        embeddings.append(padded_embeddings)
        
    return np.vstack(embeddings) if embeddings else np.empty((0, 768))

print("Embedding functions defined.")

In [ ]:
# 4. Processing Pipeline

# Placeholder: In production, load from 'raw_sequences.parquet'
# df = pl.read_parquet("raw_sequences.parquet")
# Determine sequences column name.

# Creating synthetic data for testing
sequences = [
    "ATGCTAGCTAGCTGATGCTAGCTAGCTAGCTGATCGATCGATCGATCGATCGCGCTAGCTAGC",
    "GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGC",
    "TACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTA"
] * 10 

print(f"Processing {len(sequences)} sequences...")

# Generate Embeddings (512D -> Padded to 768D)
embeddings_768d = get_embeddings(sequences, batch_size=4)

print(f"Generated embeddings shape: {embeddings_768d.shape}")

# Create DataFrame for LanceDB
data = []
for i, seq in enumerate(sequences):
    data.append({
        "vector": embeddings_768d[i],
        "sequence": seq,
        "id": f"seq_{i}",
        "metadata": {"source": "synthetic"}
    })

# Save to LanceDB
db = lancedb.connect("embeddings.lance")
# Create table (overwrite if exists)
try:
    tbl = db.create_table("sequences", data=data, mode="overwrite")
    print("LanceDB table 'sequences' created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")

# Verify
print(f"Table count: {tbl.count_rows()}")
print(tbl.head().to_pandas())